# Streaming Vocos export (clean)

This notebook keeps only inference and export steps for Streaming Vocos with Kokoro:
- clean one-shot inference path
- text-to-audio playback in notebook
- ONNX export
- LiteRT (TFLite) export


In [1]:
import math
import os
from pathlib import Path

import onnx
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.io.wavfile as wavfile
from IPython.display import Audio, display

try:
    import ai_edge_torch
except Exception:
    ai_edge_torch = None

from kokoro import KModel, KPipeline
from kokoro.model import KModelForONNX
from streaming_vocos import StreamingVocos

os.environ['CUDA_VISIBLE_DEVICES'] = ''
torch.set_num_threads(8)

OPSET_VERSION = 19
MAX_INPUT_LENGTH = 510
SAMPLE_RATE = 24000
CONFIG_FILE = 'checkpoints/config.json'
CHECKPOINT_PATH = 'checkpoints/kokoro-v1_0.pth'
VOICE_PATH = 'checkpoints/voices/af_heart.pt'
VOCOS_CKPT = 'vocos_fp16.pt'
VOCOS_HOP_LENGTH = 300
VOCOS_CHUNK_MS = 100
VOCOS_CHUNK_FRAMES = max(1, round((VOCOS_CHUNK_MS / 1000.0) * SAMPLE_RATE / VOCOS_HOP_LENGTH))
VOCOS_CHUNK_SAMPLES = VOCOS_CHUNK_FRAMES * VOCOS_HOP_LENGTH
EXPORT_DIR = Path('onnx_streaming_vocos')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


2026-03-07 17:25:57.691260: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 17:25:57.752508: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/.venv/lib/python3.12/site-packages/torch/distributed/distributed_c10d.py:351: UserWarning: Device capability of jax unspecified, assuming `cpu` and `cuda`. Please specify it via the `devices` argument of `register_backend`.
  warnings.warn(


In [2]:
# Load base model
kmodel = KModel(config=CONFIG_FILE, model=CHECKPOINT_PATH, disable_complex=True).to('cpu')
model = KModelForONNX(kmodel).eval()


def load_input_ids(pipeline: KPipeline, text: str):
    if pipeline.lang_code in 'ab':
        _, tokens = pipeline.g2p(text)
        for _, phonemes, _ in pipeline.en_tokenize(tokens):
            if not phonemes:
                continue
    else:
        phonemes, _ = pipeline.g2p(text)

    phonemes = phonemes[:MAX_INPUT_LENGTH]
    token_ids = [pipeline.model.vocab.get(p) for p in phonemes]
    token_ids = [i for i in token_ids if i is not None]
    input_ids = torch.LongTensor([[0, *token_ids, 0]]).to(pipeline.model.device)
    return phonemes, input_ids


def load_voice(pipeline: KPipeline, voice_path: str, phonemes):
    pack = pipeline.load_voice(voice_path).to('cpu')
    return pack[len(phonemes) - 1]


def prepare_inputs(model_for_onnx: KModelForONNX, text: str, voice_path: str, speed_value: int = 1):
    pipeline = KPipeline(lang_code='a', model=model_for_onnx.kmodel, device='cpu')
    phonemes, input_ids_raw = load_input_ids(pipeline, text)
    style = load_voice(pipeline, voice_path, phonemes)
    speed = torch.IntTensor([speed_value])

    text_mask = torch.zeros(1, MAX_INPUT_LENGTH, dtype=torch.float32)
    text_mask[0, :input_ids_raw.shape[1]] = 1
    input_ids = torch.nn.functional.pad(input_ids_raw, (0, MAX_INPUT_LENGTH - input_ids_raw.shape[1]))

    return {
        'phonemes': phonemes,
        'input_ids_raw': input_ids_raw,
        'input_ids': input_ids,
        'text_mask': text_mask,
        'style': style,
        'speed': speed,
    }


/rhome/eingerman/Projects/DeepLearning/TTS/Kokoro/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [3]:
class VocosChunkModule(nn.Module):
    """Fixed-size chunk vocoder module for export/runtime loops."""

    def __init__(self, chunk_frames: int = VOCOS_CHUNK_FRAMES):
        super().__init__()
        self.chunk_frames = int(chunk_frames)
        self.vocos = StreamingVocos.from_checkpoint(
            checkpoint_path=VOCOS_CKPT,
            chunk_frames=self.chunk_frames,
            device='cpu',
            use_fp16=False,
        )

    def _build_features(self, asr_chunk, f0_chunk, n_chunk, style_128):
        batch, _, t_asr = asr_chunk.shape
        t_f0 = f0_chunk.shape[-1]

        if t_asr != t_f0:
            asr_chunk = F.interpolate(asr_chunk.float(), size=t_f0, mode='linear', align_corners=False)

        f0 = f0_chunk.unsqueeze(1).float()
        n = n_chunk.unsqueeze(1).float()
        style = style_128.unsqueeze(-1).expand(batch, -1, t_f0)
        return torch.cat([asr_chunk, f0, n, style], dim=1)

    def forward(self, asr_chunk, f0_chunk, n_chunk, style_128):
        # Inputs are expected to be fixed-length chunks: T == self.chunk_frames.
        features = self._build_features(asr_chunk, f0_chunk, n_chunk, style_128)
        generator = self.vocos.model
        dtype = next(generator.parameters()).dtype
        audio = generator(features.to(dtype=dtype))
        if audio.ndim == 3 and audio.shape[1] == 1:
            audio = audio[:, 0, :]
        return audio


class BertEncoderModule(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.bert = base_model.bert
        self.bert_encoder = base_model.bert_encoder

    def forward(self, input_ids, text_mask):
        bert_dur = self.bert(input_ids, attention_mask=text_mask)
        return self.bert_encoder(bert_dur).transpose(-1, -2)


class DurationPredictorModule(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.predictor = base_model.predictor
        self.text_encoder = base_model.text_encoder

    def forward(self, input_ids, d_en, style, text_mask, speed):
        d = self.predictor.text_encoder(d_en, style[:, 128:], text_mask)
        x, _ = self.predictor.lstm(d)

        duration = self.predictor.duration_proj(x)
        duration = text_mask * torch.sigmoid(duration).sum(axis=-1) / speed
        pred_dur = torch.round(duration).squeeze()

        boundaries = torch.cumsum(pred_dur, dim=0)
        values = torch.arange(boundaries[-1], device=pred_dur.device)
        expanded_indices = torch.sum(boundaries.unsqueeze(1) <= values.unsqueeze(0), dim=0)

        en = torch.index_select(d.transpose(-1, -2), 2, expanded_indices)
        en, _ = self.predictor.shared(en.transpose(-1, -2))

        t_en = self.text_encoder(input_ids, text_mask)
        asr = torch.index_select(t_en, 2, expanded_indices)
        return pred_dur, en, asr


class F0NPredictorModule(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.predictor = base_model.predictor

    def forward(self, en, style):
        f0_pred, n_pred = self.predictor.F0Ntrain(en, style[:, 128:256])
        return f0_pred, n_pred


@torch.no_grad()
def vocode_in_fixed_chunks(vocoder_chunk_module, asr, f0_pred, n_pred, style_128):
    """Looped chunk vocoding with end padding + invalid-tail trimming."""
    total_frames = int(f0_pred.shape[-1])
    chunk_frames = vocoder_chunk_module.chunk_frames
    hop = VOCOS_HOP_LENGTH

    chunks = []
    pos = 0
    while pos < total_frames:
        end = min(total_frames, pos + chunk_frames)
        valid_frames = int(end - pos)

        asr_chunk = asr[:, :, pos:end]
        f0_chunk = f0_pred[:, pos:end]
        n_chunk = n_pred[:, pos:end]

        if valid_frames < chunk_frames:
            pad = chunk_frames - valid_frames
            asr_chunk = F.pad(asr_chunk, (0, pad))
            f0_chunk = F.pad(f0_chunk, (0, pad))
            n_chunk = F.pad(n_chunk, (0, pad))

        audio_chunk = vocoder_chunk_module(asr_chunk, f0_chunk, n_chunk, style_128)
        # Keep only valid audio samples from the padded last chunk.
        chunks.append(audio_chunk[..., : valid_frames * hop])
        pos = end

    return torch.cat(chunks, dim=-1).squeeze(0)


bert_module = BertEncoderModule(model.kmodel).eval()
duration_module = DurationPredictorModule(model.kmodel).eval()
f0n_module = F0NPredictorModule(model.kmodel).eval()
vocoder_chunk_module = VocosChunkModule(chunk_frames=VOCOS_CHUNK_FRAMES).eval()


@torch.no_grad()
def run_inference(input_ids, text_mask, style, speed):
    d_en = bert_module(input_ids, text_mask)
    pred_dur, en, asr = duration_module(input_ids, d_en, style, text_mask, speed)
    f0_pred, n_pred = f0n_module(en, style)
    audio = vocode_in_fixed_chunks(vocoder_chunk_module, asr, f0_pred, n_pred, style[:, 0:128])
    return {
        'd_en': d_en,
        'pred_dur': pred_dur,
        'en': en,
        'asr': asr,
        'audio': audio,
        'f0_pred': f0_pred,
        'n_pred': n_pred,
    }


In [4]:
# Test inference on input text and play audio inline
TEXT = (
    'I had returned to civil practice and had finally abandoned Holmes in his '
    'Baker Street rooms, although I continually visited him and occasionally '
    'even persuaded him to forgo his Bohemian habits so far as to come and visit us.'
)

inputs = prepare_inputs(model, text=TEXT, voice_path=VOICE_PATH, speed_value=1)
outputs = run_inference(
    inputs['input_ids'],
    inputs['text_mask'],
    inputs['style'],
    inputs['speed'],
)

audio_tensor = outputs['audio'].detach().cpu().float()
audio_np = audio_tensor.numpy()

display(Audio(audio_np, rate=SAMPLE_RATE))

wav_path = EXPORT_DIR / 'sample_streaming_vocos.wav'
wavfile.write(str(wav_path), SAMPLE_RATE, (audio_np * 32767).astype('int16'))

print(f'phonemes={len(inputs["phonemes"])}')
print(f'audio_shape={tuple(audio_tensor.shape)}')
print(f'saved={wav_path}')


RuntimeError: Input and output sizes should be greater than 0, but got input (W: 0) and output (W: 8)

In [ ]:
def _first_vocoder_chunk(intermediate_tensors, style):
    asr = intermediate_tensors['asr']
    f0 = intermediate_tensors['f0_pred']
    n = intermediate_tensors['n_pred']

    valid_frames = min(int(asr.shape[-1]), VOCOS_CHUNK_FRAMES)
    asr_chunk = asr[:, :, :valid_frames]
    f0_chunk = f0[:, :valid_frames]
    n_chunk = n[:, :valid_frames]

    if valid_frames < VOCOS_CHUNK_FRAMES:
        pad = VOCOS_CHUNK_FRAMES - valid_frames
        asr_chunk = F.pad(asr_chunk, (0, pad))
        f0_chunk = F.pad(f0_chunk, (0, pad))
        n_chunk = F.pad(n_chunk, (0, pad))

    return asr_chunk, f0_chunk, n_chunk, style[:, 0:128]


def export_onnx_modules(export_dir: Path, input_tensors, intermediate_tensors):
    export_dir.mkdir(parents=True, exist_ok=True)

    input_ids = input_tensors['input_ids']
    text_mask = input_tensors['text_mask']
    style = input_tensors['style']
    speed = input_tensors['speed']

    d_en = intermediate_tensors['d_en']
    en = intermediate_tensors['en']

    bert_onnx = export_dir / 'bert.onnx'
    torch.onnx.export(
        bert_module,
        args=(input_ids, text_mask),
        f=str(bert_onnx),
        input_names=['input_ids', 'text_mask'],
        output_names=['d_en'],
        opset_version=OPSET_VERSION,
        do_constant_folding=True,
        dynamo=True,
        external_data=False,
    )
    onnx.checker.check_model(onnx.load(str(bert_onnx)))

    duration_onnx = export_dir / 'duration_predictor.onnx'
    torch.onnx.export(
        duration_module,
        args=(input_ids, d_en, style, text_mask, speed),
        f=str(duration_onnx),
        input_names=['input_ids', 'd_en', 'style', 'text_mask', 'speed'],
        output_names=['pred_dur', 'en', 'asr'],
        opset_version=OPSET_VERSION,
        do_constant_folding=True,
        dynamo=False,
        external_data=False,
    )
    onnx.checker.check_model(onnx.load(str(duration_onnx)))

    f0n_onnx = export_dir / 'f0n_predictor.onnx'
    torch.onnx.export(
        f0n_module,
        args=(en, style),
        f=str(f0n_onnx),
        input_names=['en', 'style'],
        output_names=['f0_pred', 'n_pred'],
        opset_version=OPSET_VERSION,
        do_constant_folding=True,
        dynamo=False,
        external_data=False,
    )
    onnx.checker.check_model(onnx.load(str(f0n_onnx)))

    asr_chunk, f0_chunk, n_chunk, style_128 = _first_vocoder_chunk(intermediate_tensors, style)
    vocoder_onnx = export_dir / 'vocoder_chunk.onnx'
    torch.onnx.export(
        vocoder_chunk_module,
        args=(asr_chunk, f0_chunk, n_chunk, style_128),
        f=str(vocoder_onnx),
        input_names=['asr_chunk', 'f0_chunk', 'n_chunk', 'style_128'],
        output_names=['audio_chunk'],
        opset_version=OPSET_VERSION,
        do_constant_folding=True,
        dynamo=False,
        external_data=False,
    )
    onnx.checker.check_model(onnx.load(str(vocoder_onnx)))

    return {
        'bert': bert_onnx,
        'duration_predictor': duration_onnx,
        'f0n_predictor': f0n_onnx,
        'vocoder_chunk': vocoder_onnx,
    }


onnx_paths = export_onnx_modules(EXPORT_DIR, inputs, outputs)
print('ONNX export complete:')
print(f'chunk_frames={VOCOS_CHUNK_FRAMES} chunk_samples={VOCOS_CHUNK_SAMPLES}')
for name, path in onnx_paths.items():
    print(f'  {name}: {path}')


In [ ]:
def export_litert_modules(export_dir: Path, input_tensors, intermediate_tensors):
    if ai_edge_torch is None:
        raise ImportError('ai_edge_torch is not installed in this environment.')

    export_dir.mkdir(parents=True, exist_ok=True)

    input_ids = input_tensors['input_ids']
    text_mask = input_tensors['text_mask']
    style = input_tensors['style']
    speed = input_tensors['speed']

    d_en = intermediate_tensors['d_en']
    en = intermediate_tensors['en']

    bert_tflite = ai_edge_torch.convert(
        bert_module,
        sample_args=(input_ids, text_mask),
        strict_export=False,
    )
    bert_path = export_dir / 'bert.tflite'
    bert_tflite.export(str(bert_path))

    duration_tflite = ai_edge_torch.convert(
        duration_module,
        sample_args=(input_ids, d_en, style, text_mask, speed),
        strict_export=False,
    )
    duration_path = export_dir / 'duration_predictor.tflite'
    duration_tflite.export(str(duration_path))

    f0n_tflite = ai_edge_torch.convert(
        f0n_module,
        sample_args=(en, style),
        strict_export=False,
    )
    f0n_path = export_dir / 'f0n_predictor.tflite'
    f0n_tflite.export(str(f0n_path))

    asr_chunk, f0_chunk, n_chunk, style_128 = _first_vocoder_chunk(intermediate_tensors, style)
    vocoder_tflite = ai_edge_torch.convert(
        vocoder_chunk_module,
        sample_args=(asr_chunk, f0_chunk, n_chunk, style_128),
        strict_export=False,
    )
    vocoder_path = export_dir / 'vocoder_chunk.tflite'
    vocoder_tflite.export(str(vocoder_path))

    return {
        'bert': bert_path,
        'duration_predictor': duration_path,
        'f0n_predictor': f0n_path,
        'vocoder_chunk': vocoder_path,
    }


litert_paths = export_litert_modules(EXPORT_DIR, inputs, outputs)
print('LiteRT export complete:')
print(f'chunk_frames={VOCOS_CHUNK_FRAMES} chunk_samples={VOCOS_CHUNK_SAMPLES}')
for name, path in litert_paths.items():
    print(f'  {name}: {path}')
